In [ ]:
import time
start_time = time.time()

import pandas as pd
import torch
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


In [ ]:
if torch.backends.mps.is_available():
    torch_device = "mps"
    pipeline_device = "mps"
elif torch.cuda.is_available():
    torch_device = "cuda"
    pipeline_device = 0
else:
    torch_device = "cpu"
    pipeline_device = -1

print(f"Selected torch device: {torch_device}")
print(f"Pipeline device setting: {pipeline_device}")


In [ ]:
model_name = "textattack/distilbert-base-uncased-MRPC"
clf = pipeline(
    task="text-classification",
    model=model_name,
    tokenizer=model_name,
    device=pipeline_device
)
print(f"Loaded pipeline model: {model_name}")
print(clf.model.config.id2label)


In [ ]:
subset_size = 64
dataset = load_dataset("glue", "mrpc", split=f"validation[:{subset_size}]")
print(f"Validation subset examples: {len(dataset)}")

preview_df = dataset.to_pandas()[["sentence1", "sentence2", "label"]].copy()
print(preview_df.head(5).to_string(index=False))


In [ ]:
batch_size = 16
pairs = [{"text": ex["sentence1"], "text_pair": ex["sentence2"]} for ex in dataset]
raw_outputs = clf(pairs, batch_size=batch_size, truncation=True)

label_to_id = {str(v).upper(): int(k) for k, v in clf.model.config.id2label.items()}

if "LABEL_0" in label_to_id and "LABEL_1" in label_to_id:
    pass
elif "NOT_EQUIVALENT" in label_to_id and "EQUIVALENT" in label_to_id:
    label_to_id = {"LABEL_0": label_to_id["NOT_EQUIVALENT"], "LABEL_1": label_to_id["EQUIVALENT"]}
elif "PARAPHRASE" in label_to_id and "NOT_PARAPHRASE" in label_to_id:
    label_to_id = {"LABEL_0": label_to_id["NOT_PARAPHRASE"], "LABEL_1": label_to_id["PARAPHRASE"]}
else:
    raise ValueError(f"Unexpected label mapping: {clf.model.config.id2label}")

predictions = []
for out in raw_outputs:
    pred = label_to_id[out["label"].upper()]
    predictions.append(pred)

true_labels = dataset["label"]
print(f"Completed inference for {len(predictions)} examples.")
print(f"First 5 raw outputs: {raw_outputs[:5]}")


In [ ]:
accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels,
    predictions,
    average="binary",
    zero_division=0
)
cm = confusion_matrix(true_labels, predictions)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("Confusion Matrix:")
print(cm)

results_df = pd.DataFrame([
    {
        "model_name": model_name,
        "split": f"validation[:{subset_size}]",
        "num_examples": len(dataset),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "torch_device": torch_device,
        "pipeline_device": str(pipeline_device)
    }
])

print(results_df.to_string(index=False))


In [ ]:
examples_df = dataset.to_pandas()[["sentence1", "sentence2", "label"]].copy()
examples_df = examples_df.rename(columns={"label": "true_label"})
examples_df["predicted_label"] = predictions
examples_df["is_correct"] = examples_df["true_label"] == examples_df["predicted_label"]
examples_df["prediction_score"] = [float(x["score"]) for x in raw_outputs]
examples_df["pipeline_label"] = [x["label"] for x in raw_outputs]

print(examples_df.head(10).to_string(index=False))

correct_df = examples_df[examples_df["is_correct"]].copy()
incorrect_df = examples_df[~examples_df["is_correct"]].copy()

print(f"\nCorrect predictions: {len(correct_df)}")
if len(correct_df) > 0:
    print(correct_df.head(10).to_string(index=False))

print(f"\nIncorrect predictions: {len(incorrect_df)}")
if len(incorrect_df) > 0:
    print(incorrect_df.head(10).to_string(index=False))


In [ ]:
elapsed_seconds = time.time() - start_time
print(f"Total runtime (seconds): {elapsed_seconds:.2f}")
